In [1]:
import os
from pathlib import Path

import pandas as pd

# Configure pandas to show all columns and data without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
import time


def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell variables first, then fall back to the local .env file so the notebook works in VS Code and headless runs.
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default


def get_env_int(key, default):
    return int(get_env_value(key, str(default)))


def get_env_float(key, default):
    return float(get_env_value(key, str(default)))


# Keep the notebook data paths configurable so the same code works across local machines and test setups.
csv_path = Path(get_env_value("CSV_PATH", "IOT Data Simulation/smart_logistic_tracker_japan.csv"))
sample_rows = get_env_int("SAMPLE_ROWS", 5)
write_delay_seconds = get_env_float("WRITE_DELAY_SECONDS", 0.1)
write_gas_limit = get_env_int("WRITE_GAS_LIMIT", 3000000)
enable_duplicate_writes = get_env_value("ENABLE_DUPLICATE_WRITES", "false").lower() in {"1", "true", "yes", "on"}
abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load the CSV file with basic error handling
try:
    df = pd.read_csv(csv_path)
    print(f"Total records in CSV: {len(df)}")
    print(f"First {sample_rows} records:")

    # Display the first few rows
    display(df.head(sample_rows))
except FileNotFoundError:
    print(f"❌ CSV file not found: {csv_path}")
    df = pd.DataFrame()
except pd.errors.EmptyDataError:
    print(f"❌ CSV file is empty: {csv_path}")
    df = pd.DataFrame()
except pd.errors.ParserError as error:
    print(f"❌ Failed to parse CSV file {csv_path}: {error}")
    df = pd.DataFrame()
except Exception as error:
    print(f"❌ Unexpected error while loading {csv_path}: {error}")
    df = pd.DataFrame()

Total records in CSV: 100
First 5 records:


,timestamp,carrier,tracking_number,package_id,origin,current_location,delivery_location,prefecture,latitude,longitude,latest_status,logistics_delay_reason,logistics_delay,order_date,expected_delivery_date,waiting_time_minutes,perishable,temperature,humidity,rfid_tag,rfid_verified,tamper_alert,traffic_status,inventory_level,asset_utilization
0,2026-05-04 13:50:26.857905,Yamato Transport,942646961460,PKG7545,Tokyo,Naha Central Post Office,Tokyo,Kanagawa,35.993159,139.038781,Out for Delivery,NaN,0,2026-04-29 23:26:26.858009,2026-05-05 23:26:26.858015,45,No,10.7,40,RFID736892,False,No,Heavy,99,84.59
1,2026-05-03 23:36:26.858095,Japan Post,74355111775,PKG2659,Tokyo,Nagoya Central Post Office,Kyoto,Kanagawa,35.691292,139.130870,Arrival,NaN,0,2026-04-28 23:26:26.858141,2026-05-07 23:26:26.858146,54,Yes,6.2,82,RFID156229,False,Yes,Detour,363,53.39
2,2026-05-04 08:32:26.858217,Japan Post,217497030475,PKG7965,Osaka,Nagoya Central Post Office,Osaka,Aichi,35.591109,139.784940,Storage,NaN,1,2026-05-03 23:26:26.858267,2026-05-08 23:26:26.858271,144,Yes,-3.1,86,RFID890703,True,Yes,Heavy,25,95.75
3,2026-05-04 02:35:26.858332,Japan Post,249781996688,PKG5296,Fukuoka,Sapporo Central Post Office,Sapporo,Osaka,35.570440,139.689163,Delivered to the delivery address,NaN,0,2026-05-03 23:26:26.858370,2026-05-05 23:26:26.858374,173,Yes,6.3,87,RFID603182,True,Yes,Detour,145,63.84
4,2026-05-04 08:28:26.858444,Japan Post,718415724062,PKG9987,Fukuoka,Yokohama Sales Office,Sapporo,Hokkaido,35.679375,139.408071,Bring it back due to your absence,Address Unknown,1,2026-05-03 23:26:26.858492,2026-05-07 23:26:26.858501,82,No,0.7,60,RFID921432,False,Yes,Detour,34,56.28


In [2]:
from web3 import Web3

# Connect to local blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [3]:
import json

# Use the loaded ABI path and deployed contract address.
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

# Load the ABI that matches the deployed contract in this repository.
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Load the smart contract.
contract = web3.eth.contract(address=contract_address, abi=abi)

# Ganache may expose a different unlocked account set than the deployed contract owner,
# so we fall back to an explicit override when needed.
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if not override_owner:
        raise ValueError(
            f"Contract owner {contract_owner} is not unlocked in Ganache. "
            "Set CONTRACT_OWNER in .env to an unlocked account."
        )
    contract_owner = Web3.to_checksum_address(override_owner)
    if contract_owner not in web3.eth.accounts:
        raise ValueError(
            f"CONTRACT_OWNER {contract_owner} is not unlocked in Ganache."
        )

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [4]:
# Retrieve all existing records from the blockchain once at startup to cache them.
# This prevents expensive O(N) blockchain roundtrips for duplicate verification on every iteration.
total_records = contract.functions.getTotalRecords().call()
existing_records = set()
for record_index in range(total_records):
    record = contract.functions.getRecord(record_index).call()
    existing_records.add((str(record[1]), str(record[2]), str(record[3])))

def record_exists(package_id, data_type, data_value):
    """Return True when the exact record is already in the cache."""
    return (str(package_id), str(data_type), str(data_value)) in existing_records


def send_iot_data(package_id, data_type, data_value):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    # Skip exact duplicates unless testing explicitly requires them.
    if not enable_duplicate_writes and record_exists(package_id, data_type, data_value):
        print(
            f"ℹ️ Skipped duplicate | {package_id} | "
            f"Type: {data_type} | Value: {data_value}"
        )
        return False

    if enable_duplicate_writes:
        print("ℹ️ Duplicate-write mode is ON; exact duplicates will be stored.")

    txn = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': write_gas_limit
    })

    # Wait for transaction confirmation before moving to the next record.
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    # Cache the new entry locally
    existing_records.add((str(package_id), str(data_type), str(data_value)))

    print(
        f"✅ Data Stored | {package_id} | "
        f"Type: {data_type} | "
        f"Value: {data_value} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )
    return True

# Each CSV row writes multiple contract entries depending on the columns.
is_iot_data = "shipment_id" in df.columns
entries_per_row = 3 if is_iot_data else 4

target_contract_records = int(get_env_value("TARGET_CONTRACT_RECORDS", "100"))
target_rows = target_contract_records // entries_per_row
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records
rows_to_store = min(len(df), target_rows, remaining_entries // entries_per_row)

print(f"Target contract records: {target_contract_records}")
print(f"Current records: {current_records}")
print(f"Maximum records: {max_entries}")
print(f"Remaining contract slots: {remaining_entries}")
print(f"Rows that can still be stored safely: {rows_to_store}")
print(f"Duplicate writes enabled: {enable_duplicate_writes}")

if target_contract_records % entries_per_row != 0:
    print(f"⚠️ TARGET_CONTRACT_RECORDS is not a multiple of {entries_per_row}, ignoring remaining slots to keep rows complete.")

if rows_to_store <= 0:
    print("⚠️ No remaining storage capacity on the contract.")
else:
    stored_rows = 0
    skipped_rows = 0

    for index, row in df.head(rows_to_store).iterrows():
        if is_iot_data:
            package_id = str(row["shipment_id"])
            status = str(row["shipment_status"])
            temp = f"{row['temperature']}°C"
            humid = f"{row['humidity']}%"

            s1 = send_iot_data(package_id, "Status", status)
            s2 = send_iot_data(package_id, "Temperature", temp)
            s3 = send_iot_data(package_id, "Humidity", humid)

            if s1 or s2 or s3:
                stored_rows += 1
            else:
                skipped_rows += 1
        else:
            package_id = str(row["package_id"])
            location = str(row["current_location"])
            status = str(row["latest_status"])
            temp = f"{row['temperature']}°C"
            humid = f"{row['humidity']}%"

            s1 = send_iot_data(package_id, "Location", location)
            s2 = send_iot_data(package_id, "Status", status)
            s3 = send_iot_data(package_id, "Temperature", temp)
            s4 = send_iot_data(package_id, "Humidity", humid)

            if s1 or s2 or s3 or s4:
                stored_rows += 1
            else:
                skipped_rows += 1

        # Small pause between transactions keeps Ganache logs readable and avoids flooding the provider.
        time.sleep(write_delay_seconds)

    print(f"\n✅ Successfully stored {stored_rows} new rows on the blockchain!")
    if skipped_rows:
        print(f"ℹ️ Skipped {skipped_rows} duplicate rows.")

Target contract records: 100
Current records: 315
Maximum records: 500
Remaining contract slots: 185
Rows that can still be stored safely: 25
Duplicate writes enabled: False
ℹ️ Skipped duplicate | PKG7545 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG7545 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | PKG7545 | Type: Temperature | Value: 10.7°C
ℹ️ Skipped duplicate | PKG7545 | Type: Humidity | Value: 40%
ℹ️ Skipped duplicate | PKG2659 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG2659 | Type: Status | Value: Arrival
ℹ️ Skipped duplicate | PKG2659 | Type: Temperature | Value: 6.2°C
ℹ️ Skipped duplicate | PKG2659 | Type: Humidity | Value: 82%


ℹ️ Skipped duplicate | PKG7965 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG7965 | Type: Status | Value: Storage
ℹ️ Skipped duplicate | PKG7965 | Type: Temperature | Value: -3.1°C
ℹ️ Skipped duplicate | PKG7965 | Type: Humidity | Value: 86%
ℹ️ Skipped duplicate | PKG5296 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG5296 | Type: Status | Value: Delivered to the delivery address
ℹ️ Skipped duplicate | PKG5296 | Type: Temperature | Value: 6.3°C
ℹ️ Skipped duplicate | PKG5296 | Type: Humidity | Value: 87%


ℹ️ Skipped duplicate | PKG9987 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG9987 | Type: Status | Value: Bring it back due to your absence
ℹ️ Skipped duplicate | PKG9987 | Type: Temperature | Value: 0.7°C
ℹ️ Skipped duplicate | PKG9987 | Type: Humidity | Value: 60%
ℹ️ Skipped duplicate | PKG8392 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG8392 | Type: Status | Value: Hold at Yamato
ℹ️ Skipped duplicate | PKG8392 | Type: Temperature | Value: 2.5°C
ℹ️ Skipped duplicate | PKG8392 | Type: Humidity | Value: 61%


ℹ️ Skipped duplicate | PKG2808 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG2808 | Type: Status | Value: Delay
ℹ️ Skipped duplicate | PKG2808 | Type: Temperature | Value: 0.9°C
ℹ️ Skipped duplicate | PKG2808 | Type: Humidity | Value: 48%
ℹ️ Skipped duplicate | PKG1749 | Type: Location | Value: Fukuoka Distribution Center
ℹ️ Skipped duplicate | PKG1749 | Type: Status | Value: Arrival
ℹ️ Skipped duplicate | PKG1749 | Type: Temperature | Value: 19.6°C
ℹ️ Skipped duplicate | PKG1749 | Type: Humidity | Value: 34%


ℹ️ Skipped duplicate | PKG1803 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG1803 | Type: Status | Value: Hand it over at the window
ℹ️ Skipped duplicate | PKG1803 | Type: Temperature | Value: 22.6°C
ℹ️ Skipped duplicate | PKG1803 | Type: Humidity | Value: 55%
ℹ️ Skipped duplicate | PKG2151 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG2151 | Type: Status | Value: In Transit
ℹ️ Skipped duplicate | PKG2151 | Type: Temperature | Value: 23.0°C
ℹ️ Skipped duplicate | PKG2151 | Type: Humidity | Value: 89%


...

ℹ️ Skipped duplicate | PKG8850 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG8850 | Type: Status | Value: Delivered
ℹ️ Skipped duplicate | PKG8850 | Type: Temperature | Value: 11.1°C
ℹ️ Skipped duplicate | PKG8850 | Type: Humidity | Value: 51%
ℹ️ Skipped duplicate | PKG8659 | Type: Location | Value: Fukuoka Distribution Center
ℹ️ Skipped duplicate | PKG8659 | Type: Status | Value: Hold at Yamato
ℹ️ Skipped duplicate | PKG8659 | Type: Temperature | Value: 21.1°C
ℹ️ Skipped duplicate | PKG8659 | Type: Humidity | Value: 88%


ℹ️ Skipped duplicate | PKG1347 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG1347 | Type: Status | Value: Returned to the sender
ℹ️ Skipped duplicate | PKG1347 | Type: Temperature | Value: 18.8°C
ℹ️ Skipped duplicate | PKG1347 | Type: Humidity | Value: 76%
ℹ️ Skipped duplicate | PKG8088 | Type: Location | Value: Tokyo Central Post Office
ℹ️ Skipped duplicate | PKG8088 | Type: Status | Value: Under Investigation
ℹ️ Skipped duplicate | PKG8088 | Type: Temperature | Value: 8.7°C
ℹ️ Skipped duplicate | PKG8088 | Type: Humidity | Value: 37%


ℹ️ Skipped duplicate | PKG2762 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG2762 | Type: Status | Value: Arrival Scan
ℹ️ Skipped duplicate | PKG2762 | Type: Temperature | Value: -2.5°C
ℹ️ Skipped duplicate | PKG2762 | Type: Humidity | Value: 77%
ℹ️ Skipped duplicate | PKG8577 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG8577 | Type: Status | Value: Hand it over at the window
ℹ️ Skipped duplicate | PKG8577 | Type: Temperature | Value: 0.7°C
ℹ️ Skipped duplicate | PKG8577 | Type: Humidity | Value: 36%


ℹ️ Skipped duplicate | PKG5517 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG5517 | Type: Status | Value: Delay
ℹ️ Skipped duplicate | PKG5517 | Type: Temperature | Value: -2.4°C
ℹ️ Skipped duplicate | PKG5517 | Type: Humidity | Value: 85%
ℹ️ Skipped duplicate | PKG7409 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG7409 | Type: Status | Value: Arrival
ℹ️ Skipped duplicate | PKG7409 | Type: Temperature | Value: 2.3°C
ℹ️ Skipped duplicate | PKG7409 | Type: Humidity | Value: 65%


ℹ️ Skipped duplicate | PKG5985 | Type: Location | Value: Tokyo Central Post Office
ℹ️ Skipped duplicate | PKG5985 | Type: Status | Value: Returned
ℹ️ Skipped duplicate | PKG5985 | Type: Temperature | Value: 4.3°C
ℹ️ Skipped duplicate | PKG5985 | Type: Humidity | Value: 61%

✅ Successfully stored 0 new rows on the blockchain!
ℹ️ Skipped 25 duplicate rows.


In [5]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

Total IoT records stored: 315


In [6]:
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records

print(f"Current IoT records stored: {current_records}")
print(f"Maximum records allowed: {max_entries}")
print(f"Remaining storage slots: {remaining_entries}")

if remaining_entries == 0:
    print("⚠️ The contract is full. Redeploy a new contract or reset the chain to store more data.")
elif remaining_entries <= 20:
    print("⚠️ The contract is nearing capacity.")

Current IoT records stored: 315
Maximum records allowed: 500
Remaining storage slots: 185


In [7]:
# Retrieve and display the first stored record
first_record = contract.functions.getRecord(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Data Type: {first_record[2]}")
print(f"Data Value: {first_record[3]}")

📦 First Stored Record
Timestamp: 1780192485
Package ID: PKG7545
Data Type: Location
Data Value: Naha Central Post Office


In [8]:
# Retrieve and display the first 5 stored records in the blockchain
total_records = contract.functions.getTotalRecords().call()
print(f"Total records stored on blockchain: {total_records}")

records_to_show = min(5, total_records)
print(f"\n📦 First {records_to_show} Stored Records:")
for i in range(records_to_show):
    record = contract.functions.getRecord(i).call()
    print(f"Record #{i+1}:")
    print(f"  Timestamp: {record[0]}")
    print(f"  Package ID: {record[1]}")
    print(f"  Data Type: {record[2]}")
    print(f"  Data Value: {record[3]}")
    print("-" * 30)

Total records stored on blockchain: 315

📦 First 5 Stored Records:
Record #1:
  Timestamp: 1780192485
  Package ID: PKG7545
  Data Type: Location
  Data Value: Naha Central Post Office
------------------------------
Record #2:
  Timestamp: 1780192485
  Package ID: PKG7545
  Data Type: Status
  Data Value: Out for Delivery
------------------------------
Record #3:
  Timestamp: 1780192486
  Package ID: PKG2659
  Data Type: Location
  Data Value: Nagoya Central Post Office
------------------------------
Record #4:
  Timestamp: 1780192486
  Package ID: PKG2659
  Data Type: Status
  Data Value: Arrival
------------------------------
Record #5:
  Timestamp: 1780192486
  Package ID: PKG7965
  Data Type: Location
  Data Value: Nagoya Central Post Office
------------------------------
